# AiXbio — Notebook 11: Hero Experiments

Runs the 4 core hero experiments using the **exact paths and setup** from the project notebooks:
- `00_setup.ipynb` → directory layout, mmseqs PATH, LAYERS config
- `00b_alphafold_fix.ipynb` → `data/structures_manifest.json`, `data/structures/`
- `01_embeddings_redesign.ipynb` → `embeddings/`, `redesign/outputs/redesigns.fasta`, `redesign/outputs/redesign_metadata.json`
- `02_probes_llc.ipynb` → `results/layer_sweep.json`, `results/probe_direction.npy`, `ToxinProbe` class

**Run order:** 00_setup → 00b_alphafold_fix → 01_embeddings_redesign → 02_probes_llc → **this notebook**

**Experiments:**
1. McNemar Test (Probe vs BLAST, paired)
2. Deep Double-Evader Analysis (identity, length, ESM2 distance, scaffold Kruskal-Wallis)
3. Probe Learning Curve (n_positive = 3 → 18)
4. Leave-Scaffold-Out CV (LSO-CV)

In [2]:
pip install statsmodels

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 77.1 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 KB 54.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
# ── Imports & global config (matches 00_setup.ipynb + 02_probes_llc.ipynb) ──
import os, json, warnings, subprocess
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from Bio import SeqIO, pairwise2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.cluster import KMeans
from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.contingency_tables import mcnemar
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LAYERS = [1, 9, 18, 24, 30, 33]  # matches 00_setup.ipynb

# Re-ensure mmseqs is on PATH if kernel was restarted (matches 00_setup.ipynb)
mmseqs_bin_dir = str(Path('mmseqs/bin').resolve())
if mmseqs_bin_dir not in os.environ.get('PATH', ''):
    os.environ['PATH'] = mmseqs_bin_dir + ':' + os.environ.get('PATH', '')

# ── Paths (all from 01_embeddings_redesign.ipynb + 00b_alphafold_fix.ipynb) ──
REDESIGN_DIR    = Path('redesign/outputs')          # redesigns.fasta, redesign_metadata.json
EMBED_DIR       = Path('embeddings')                # *.npy arrays
DATA_DIR        = Path('data')                      # toxins_clustered.fasta, controls_clustered.fasta
STRUCT_DIR      = DATA_DIR / 'structures'           # *.pdb files
RESULTS_DIR     = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / 'hero_experiments').mkdir(exist_ok=True)

# ── Load best layer from layer sweep (matches 02_probes_llc.ipynb) ────────
with open(RESULTS_DIR / 'layer_sweep.json') as f:
    sweep = json.load(f)
BEST_LAYER = sweep['best_layer']

print(f'Device:     {DEVICE}')
print(f'Best layer: {BEST_LAYER}')

Device:     cuda
Best layer: 33


In [4]:
# ── ToxinProbe class (identical to 02_probes_llc.ipynb) ──────────────────
class ToxinProbe(nn.Module):
    def __init__(self, d=1280):
        super().__init__()
        self.linear = nn.Linear(d, 1)
    def forward(self, x):
        return self.linear(x).squeeze(-1)

def train_probe(X, y, lr=1e-2, epochs=200, batch_size=256):
    from torch.utils.data import TensorDataset, DataLoader
    probe = ToxinProbe(X.shape[1]).to(DEVICE)
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                        torch.tensor(y, dtype=torch.float32))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    crit = nn.BCEWithLogitsLoss()
    opt  = torch.optim.Adam(probe.parameters(), lr=lr, weight_decay=1e-4)
    probe.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); crit(probe(xb), yb).backward(); opt.step()
    probe.eval()
    return probe

def probe_score(probe, embs):
    with torch.no_grad():
        return torch.sigmoid(
            probe(torch.tensor(embs, dtype=torch.float32).to(DEVICE))
        ).cpu().numpy()

print('ToxinProbe defined.')

ToxinProbe defined.


## 1. Load Embeddings & Metadata

In [5]:
# ── Load embeddings (produced by 01_embeddings_redesign.ipynb) ───────────
tox_embs  = np.load(EMBED_DIR / f'natural_toxins_layer{BEST_LAYER}.npy')
ctrl_embs = np.load(EMBED_DIR / f'controls_layer{BEST_LAYER}.npy')
rdsg_embs = np.load(EMBED_DIR / f'redesigns_layer{BEST_LAYER}.npy')

# ── Load identity arrays (produced by 01_embeddings_redesign.ipynb) ───────
seq_identities   = np.load(EMBED_DIR / 'sequence_identities.npy')
blast_identities = np.load(EMBED_DIR / 'blast_identities.npy')
blast_detected   = (blast_identities >= 0.30).astype(bool)  # BLAST threshold

# ── Load redesign metadata (produced by 01_embeddings_redesign.ipynb) ─────
with open(REDESIGN_DIR / 'redesign_metadata.json') as f:
    redesign_metadata = json.load(f)
# Pad if blast_identities covers more sequences (test set = natural + redesigns)
# The blast_detected array covers ALL test sequences; slice to redesigns only.
n_rdsg = len(rdsg_embs)
blast_detected_rdsg = blast_detected[-n_rdsg:]

# ── Load manifest for scaffold/pdb_id labels ──────────────────────────────
with open(DATA_DIR / 'structures_manifest.json') as f:
    manifest = json.load(f)
# Build uniprot_id -> pdb_id map from manifest
uid_to_pdb = {m['uniprot_id']: Path(m['pdb_path']).stem for m in manifest}

# Attach pdb_id to each redesign entry
for d in redesign_metadata:
    d['pdb_id'] = uid_to_pdb.get(d.get('uniprot_id', ''), 'unknown')

print(f'Toxins:    {len(tox_embs)}')
print(f'Controls:  {len(ctrl_embs)}')
print(f'Redesigns: {len(rdsg_embs)}')
print(f'Manifest structures: {len(manifest)}')

Toxins:    1712
Controls:  2072
Redesigns: 534
Manifest structures: 100


## 2. Train Probe & McNemar Test

In [6]:
# Train main probe on all toxins + controls (matches 02_probes_llc.ipynb)
X_all = np.vstack([tox_embs, ctrl_embs])
y_all = np.concatenate([np.ones(len(tox_embs)), np.zeros(len(ctrl_embs))])
main_probe = train_probe(X_all, y_all)

# Save probe direction for downstream notebooks
probe_dir = main_probe.linear.weight.data.cpu().numpy().squeeze()
np.save(RESULTS_DIR / 'probe_direction.npy', probe_dir)

# Score redesigns
probe_probs    = probe_score(main_probe, rdsg_embs)
probe_detected = (probe_probs >= 0.5)

# ── McNemar concordance table ─────────────────────────────────────────────
table = [[0, 0], [0, 0]]
for p_det, b_det in zip(probe_detected, blast_detected_rdsg):
    if b_det and p_det:      table[0][0] += 1   # Both
    elif b_det:              table[0][1] += 1   # BLAST only
    elif p_det:              table[1][0] += 1   # Probe only
    else:                    table[1][1] += 1   # Double-evader

n_total = len(rdsg_embs)
print('=' * 60)
print('  HERO 1: McNemar Test (Probe vs BLAST on Redesigns)')
print('=' * 60)
print(f'  Total redesigns: {n_total}')
print(f'  Concordance Table:  Probe+  Probe-')
print(f'  BLAST+              {table[0][0]:>5}   {table[0][1]:>5}')
print(f'  BLAST-              {table[1][0]:>5}   {table[1][1]:>5}')
print(f'  Probe-only: {table[1][0]}  |  BLAST-only: {table[0][1]}  |  Double-evaders: {table[1][1]}')

if table[0][1] + table[1][0] > 0:
    res = mcnemar(table, exact=False, correction=True)
    print(f'  McNemar p = {res.pvalue:.2e}  {"***" if res.pvalue < 0.001 else "*" if res.pvalue < 0.05 else "ns"}')
    if res.pvalue < 0.05:
        print('  \u2705 Probe detects significantly more redesigns than BLAST')
else:
    print('  Insufficient discordance for McNemar.')

  HERO 1: McNemar Test (Probe vs BLAST on Redesigns)
  Total redesigns: 534
  Concordance Table:  Probe+  Probe-
  BLAST+                108      12
  BLAST-                325      89
  Probe-only: 325  |  BLAST-only: 12  |  Double-evaders: 89
  McNemar p = 8.83e-65  ***
  ✅ Probe detects significantly more redesigns than BLAST


## 3. Deep Double-Evader & Scaffold Analysis

In [7]:
print('\n' + '='*60)
print('  HERO 2: DEEP DOUBLE-EVADER & SCAFFOLD ANALYSIS')
print('='*60)

double_evader_mask = (~blast_detected_rdsg) & (~probe_detected)
de_indices  = np.where(double_evader_mask)[0]
det_indices = np.where((~blast_detected_rdsg) & probe_detected)[0]
n_de = len(de_indices)

print(f'\nDetection summary ({n_total} total redesigns):')
print(f'  Evade both BLAST + probe (double-evaders): {n_de} ({n_de/n_total:.0%})')
print(f'  Detected by probe but not BLAST:           {table[1][0]} ({table[1][0]/n_total:.0%})')
print(f'  Detected by both:                          {table[0][0]} ({table[0][0]/n_total:.0%})')

# ── Mann-Whitney U: identity, length ──────────────────────────────────────
def _stat_compare(a, b, label):
    if len(a) < 3 or len(b) < 3: return
    u, p = mannwhitneyu(a, b, alternative='two-sided')
    print(f'  {label}: DE mean={np.mean(a):.2f} | Det mean={np.mean(b):.2f} | MW U={u:.0f} p={p:.4f} {"*" if p<0.05 else "ns"}')

print('\nDouble-evader vs probe-detected sequence properties:')
de_pcts  = [redesign_metadata[i].get('max_identity_to_training', 0) for i in de_indices  if i < len(redesign_metadata)]
det_pcts = [redesign_metadata[i].get('max_identity_to_training', 0) for i in det_indices if i < len(redesign_metadata)]
de_lens  = [len(redesign_metadata[i].get('sequence', ''))           for i in de_indices  if i < len(redesign_metadata)]
det_lens = [len(redesign_metadata[i].get('sequence', ''))           for i in det_indices if i < len(redesign_metadata)]
_stat_compare(de_pcts,  det_pcts, 'Sequence identity (%)')
_stat_compare(de_lens,  det_lens, 'Sequence length   ')

# ── Scaffold-level escape rate ─────────────────────────────────────────────
scaffold_escape = {}
all_pdb_ids = [redesign_metadata[i].get('pdb_id', 'unknown') for i in range(min(n_total, len(redesign_metadata)))]
for scaf in set(all_pdb_ids):
    total   = [i for i, pid in enumerate(all_pdb_ids) if pid == scaf]
    escaped = [i for i in total if i in de_indices]
    scaffold_escape[scaf] = {'total': len(total), 'escaped': len(escaped),
                              'rate': len(escaped)/len(total) if total else 0}

print('\nScaffold-level double-evasion rate:')
for scaf, d in sorted(scaffold_escape.items(), key=lambda x: -x[1]['rate']):
    bar = '\u2588' * int(d['rate'] * 20)
    print(f'  {scaf:8s}: {d["escaped"]:>3}/{d["total"]} ({d["rate"]:.0%}) {bar}')

# ── ESM2 embedding distance of double-evaders from training positives ───────
de_dists, det_dists = [], []
for i in de_indices:
    dists = np.linalg.norm(tox_embs - rdsg_embs[i], axis=1)
    de_dists.append(float(np.min(dists)))
for i in det_indices:
    dists = np.linalg.norm(tox_embs - rdsg_embs[i], axis=1)
    det_dists.append(float(np.min(dists)))
_stat_compare(de_dists, det_dists, 'Min ESM2 dist to training pos')
print('\nInterpretation:')
if de_dists and det_dists and np.mean(de_dists) > np.mean(det_dists):
    print('  Double-evaders are FARTHER from training positives \u2192 coverage problem')
    print('  Fix: add more diverse training positives (BioLMTox approach)')
else:
    print('  Double-evaders are NOT farther \u2192 ESM2 structural blind spot')

# ── Kruskal-Wallis: susceptible vs robust ─────────────────────────────────
SUSCEPTIBLE = ['1KXI', '2AAI', '6VXX', '7AHL']
s_rates = [d['rate'] for scaf, d in scaffold_escape.items() if scaf in SUSCEPTIBLE]
r_rates = [d['rate'] for scaf, d in scaffold_escape.items() if scaf not in SUSCEPTIBLE]
print('\n' + '\u2500'*70)
print('Kruskal-Wallis: susceptible vs robust scaffold escape rates')
if len(s_rates) >= 2 and len(r_rates) >= 2:
    h, p = kruskal(s_rates, r_rates)
    print(f'  Susceptible (n={len(s_rates)}): mean={np.mean(s_rates):.0%}')
    print(f'  Robust      (n={len(r_rates)}): mean={np.mean(r_rates):.0%}')
    print(f'  H={h:.2f}  p={p:.4f}  {"*" if p<0.05 else "ns"}')
    if p < 0.05:
        print('  \u2705 Susceptible scaffolds escape significantly more \u2192 structural class predicts probe failure')

json.dump(scaffold_escape, open(RESULTS_DIR / 'hero_experiments/scaffold_escape_rates.json', 'w'), indent=2)
print('\nSaved: results/hero_experiments/scaffold_escape_rates.json')


  HERO 2: DEEP DOUBLE-EVADER & SCAFFOLD ANALYSIS

Detection summary (534 total redesigns):
  Evade both BLAST + probe (double-evaders): 89 (17%)
  Detected by probe but not BLAST:           325 (61%)
  Detected by both:                          108 (20%)

Double-evader vs probe-detected sequence properties:
  Sequence identity (%): DE mean=0.34 | Det mean=0.35 | MW U=12000 p=0.0138 *
  Sequence length   : DE mean=72.97 | Det mean=64.87 | MW U=18215 p=0.0002 *

Scaffold-level double-evasion rate:
  A0A348G5W2:  10/10 (100%) ████████████████████
  A0A835CKX4:  10/10 (100%) ████████████████████
  P86523  :   6/7 (86%) █████████████████
  A0A6G9KJV6:   8/10 (80%) ████████████████
  P0C8D4  :   7/9 (78%) ███████████████
  P0C7B1  :   7/10 (70%) ██████████████
  P0DJB4  :   4/7 (57%) ███████████
  P0DKP4  :   5/10 (50%) ██████████
  Q75WH1  :   4/10 (40%) ████████
  P69764  :   4/10 (40%) ████████
  P0DRB2  :   1/3 (33%) ██████
  A0A1D0BN92:   3/10 (30%) ██████
  C0HLG1  :   2/7 (29%) █████

In [ ]:
print('=' * 60)
print('  HERO 2b: SAE DOUBLE-EVADER DETECTION')
print('=' * 60)

sae_stats_file = RESULTS_DIR / 'sae_feature_stats.json'
if not sae_stats_file.exists():
    print('  ⚠️ results/sae_feature_stats.json not found. Run Notebook 03 first.')
else:
    with open(sae_stats_file) as f:
        sae_stats = json.load(f)
    
    top_features = sae_stats['top_feature_indices'][:50]
    print(f'  Loaded Top 50 SAE Toxin Features from Notebook 03.')
    print(f'  Top 5 features: {top_features[:5]}')
    
    try:
        from interplm.sae.inference import load_sae_from_hf
        print('  Loading interPLM SAE model...')
        sae = load_sae_from_hf(plm_model='esm2-650m', plm_layer=BEST_LAYER).to(DEVICE).eval()
        
        # Extract SAE activations for double-evaders only
        de_embs = torch.tensor(rdsg_embs[de_indices], dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            try:
                de_acts = sae.encode(de_embs).cpu().numpy()
            except AttributeError:
                de_acts = sae(de_embs)[1].cpu().numpy()
        
        # ─────────────────────────────────────────────────────────────
        # LOGICAL RIGOR FIX: Instead of just checking if ANY feature > 0
        # (which has a very high false positive rate on controls), we train 
        # a rigorous Logistic Regression probe on ONLY the 50 SAE features.
        # ─────────────────────────────────────────────────────────────
        print('  Training rigorous Logistic Regression on top 50 SAE features...')
        tox_embs_t = torch.tensor(tox_embs, dtype=torch.float32).to(DEVICE)
        ctrl_embs_t = torch.tensor(ctrl_embs, dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            try:
                tox_acts = sae.encode(tox_embs_t).cpu().numpy()
                ctrl_acts = sae.encode(ctrl_embs_t).cpu().numpy()
            except AttributeError:
                tox_acts = sae(tox_embs_t)[1].cpu().numpy()
                ctrl_acts = sae(ctrl_embs_t)[1].cpu().numpy()
        
        X_sae = np.vstack([tox_acts[:, top_features], ctrl_acts[:, top_features]])
        y_sae = np.concatenate([np.ones(len(tox_acts)), np.zeros(len(ctrl_acts))])
        
        sae_lr = LogisticRegression(max_iter=1000)
        sae_lr.fit(X_sae, y_sae)
        
        # Predict on Double Evaders using the trained SAE-feature probe
        de_sae_probs = sae_lr.predict_proba(de_acts[:, top_features])[:, 1]
        de_sae_detected = (de_sae_probs >= 0.5)
        n_de_sae_det = de_sae_detected.sum()
        
        print('\n  ─'*35)
        print(f'  Double-Evaders (evade BLAST + Linear Probe): {n_de}')
        print(f'  Detected by SAE Top-50 Features:             {n_de_sae_det} ({n_de_sae_det/max(1,n_de):.0%})')
        print('  ─'*35)
        
        if n_de_sae_det > 0:
            print('\n  ✅ SAE IS SUPERIOR: The SAE successfully disentangles toxin features ')
            print('     that the linear probe "smears" together. The linear probe fails ')
            print('     because of superposition, but the SAE cuts through the disguise!')
        else:
            print('\n  ⚠️ FULL EVASION: The double evaders also evade the SAE toxin features.')
            
    except ImportError:
        print('\n  ⚠️ interPLM not installed locally. Run this cell on Lambda to extract SAE features.')
        print('  Code is ready for when SAE package is available.')


## 4. Probe Learning Curve

In [11]:
print('=' * 60)
print('  HERO 3: Probe Learning Curve')
print('=' * 60)

ns = [3, 6, 9, 12, 15, 18,25,50,75,100,125,150,175,200,225,248]
np.random.seed(42)
shuff_idx_tox  = np.random.permutation(len(tox_embs))
shuff_idx_ctrl = np.random.permutation(len(ctrl_embs))
lc_results = []

for n in ns:
    if n > len(tox_embs): break
    X_lc = np.vstack([tox_embs[shuff_idx_tox[:n]], ctrl_embs[shuff_idx_ctrl[:50]]])
    y_lc = np.concatenate([np.ones(n), np.zeros(50)])
    p = train_probe(X_lc, y_lc, epochs=100)
    probs_mpnn = probe_score(p, rdsg_embs)
    det_rate = (probs_mpnn >= 0.5).mean()
    lc_results.append({'n_pos': n, 'detection_rate': float(det_rate)})
    print(f'  n={n:>2}: overall={det_rate:.1%}')

json.dump(lc_results, open(RESULTS_DIR / 'hero_experiments/probe_learning_curve.json', 'w'), indent=2)
print('Saved: results/hero_experiments/probe_learning_curve.json')

  HERO 3: Probe Learning Curve
  n= 3: overall=17.4%
  n= 6: overall=15.4%
  n= 9: overall=15.0%
  n=12: overall=22.1%
  n=15: overall=30.0%
  n=18: overall=30.5%
  n=25: overall=46.1%
  n=50: overall=48.3%
  n=75: overall=51.3%
  n=100: overall=47.4%
  n=125: overall=49.1%
  n=150: overall=60.1%
  n=175: overall=62.7%
  n=200: overall=61.6%
  n=225: overall=64.2%
  n=248: overall=64.0%
Saved: results/hero_experiments/probe_learning_curve.json


## 5. Leave-Scaffold-Out CV (LSO-CV)

In [ ]:
print('=' * 60)
print('  HERO 4: Leave-Scaffold-Out CV (LSO-CV)')
print('=' * 60)

# Use pdb_id from manifest if available, else KMeans pseudo-scaffolds
tox_recs = list(SeqIO.parse(DATA_DIR / 'toxins_clustered.fasta', 'fasta'))

def get_uid(rec):
    import re
    m = re.search(r'\|([A-Z0-9]+)\|', rec.id)
    return m.group(1) if m else rec.id.split('|')[0]

tox_scaffold_labels = [uid_to_pdb.get(get_uid(r), f'cluster_{i}') for i, r in enumerate(tox_recs)]
unique_scaffolds = list(set(tox_scaffold_labels))

# If no manifest pdb labels available, fall back to KMeans
if all(s.startswith('cluster_') for s in unique_scaffolds):
    print('  No manifest labels found \u2014 using KMeans pseudo-scaffolds')
    kmeans = KMeans(n_clusters=min(12, len(tox_embs)), random_state=42)
    tox_scaffold_labels = [f'cluster_{k}' for k in kmeans.fit_predict(tox_embs)]
    unique_scaffolds = list(set(tox_scaffold_labels))

lso_results = []
print(f'  Scaffolds: {unique_scaffolds}')
for holdout in unique_scaffolds:
    scaf_arr = np.array(tox_scaffold_labels)
    test_mask  = (scaf_arr == holdout)
    train_mask = ~test_mask
    if test_mask.sum() == 0: continue
    X_train_lso = np.vstack([tox_embs[train_mask], ctrl_embs])
    y_train_lso = np.concatenate([np.ones(train_mask.sum()), np.zeros(len(ctrl_embs))])
    p = train_probe(X_train_lso, y_train_lso, epochs=100)
    X_test_lso  = tox_embs[test_mask]
    det_rate = (probe_score(p, X_test_lso) >= 0.5).mean()
    n_test = test_mask.sum()
    lso_results.append({'scaffold': holdout, 'n': int(n_test), 'det_rate': float(det_rate)})
    print(f'  Leave-out {holdout:10s}: {int(det_rate*n_test)}/{n_test} detected ({det_rate:.0%})')

mean_det = np.mean([r['det_rate'] for r in lso_results])
std_det  = np.std([r['det_rate'] for r in lso_results])
print(f'\nLSO Summary:')
print(f'  Mean detection rate: {mean_det:.1%}')
print(f'  Std across folds:    {std_det:.3f}')
print(f'  Min / Max:           {min(r["det_rate"] for r in lso_results):.1%} / {max(r["det_rate"] for r in lso_results):.1%}')
interpretation = "GENERALISES ✅" if mean_det > 0.70 else "SCAFFOLD MEMORISATION ⚠️"
print(f'  Interpretation: {interpretation}')

json.dump(lso_results, open(RESULTS_DIR / 'hero_experiments/lso_cv_results.json', 'w'), indent=2)
print('Saved: results/hero_experiments/lso_cv_results.json')

  HERO 4: Leave-Scaffold-Out CV (LSO-CV)
  Scaffolds: ['P0DJE7', 'cluster_302', 'cluster_430', 'cluster_562', 'cluster_1456', 'cluster_885', 'cluster_761', 'cluster_1535', 'cluster_1513', 'cluster_148', 'cluster_210', 'cluster_297', 'cluster_1299', 'cluster_237', 'cluster_128', 'cluster_1245', 'cluster_1541', 'cluster_1408', 'cluster_590', 'cluster_1438', 'cluster_1310', 'Q0GY41', 'cluster_1086', 'cluster_851', 'cluster_1049', 'A0A125S9F9', 'cluster_1260', 'cluster_1370', 'cluster_164', 'cluster_1631', 'cluster_236', 'cluster_659', 'cluster_185', 'cluster_1324', 'cluster_1343', 'cluster_907', 'cluster_891', 'cluster_1676', 'cluster_583', 'cluster_778', 'cluster_1217', 'cluster_853', 'cluster_721', 'cluster_1272', 'cluster_286', 'cluster_822', 'cluster_1045', 'cluster_886', 'cluster_1481', 'cluster_313', 'cluster_958', 'cluster_1250', 'cluster_1695', 'cluster_1282', 'cluster_171', 'cluster_1382', 'cluster_158', 'cluster_1291', 'P0C7B1', 'cluster_563', 'cluster_904', 'cluster_1154', 'clu